[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C51_Data_Augmentation_Course/01_lexical/01_lexical_augmentation.ipynb)

# 01 · 词面增强（EDA 四操作 / 保护规则 / α 与数据量的交互）

目标：把 **SR / RI / RS / RD 四个操作** 从零实现，**精确量化它们的标签破坏率**，
再用保护规则把保真度从 ~90% 提到 99%+，最后复现「小数据集才受益」这条曲线。

路线：四操作实现 → **危险性排序（破坏率差一个量级）** → 保护规则与互信息自动挑关键词 →
保真-多样权衡前沿 → α 扫描（先升后急降）→ 数据量-收益曲线 → 在线 vs 离线增强 →
✏️ 练习 → 📖 答案 → 🧪 预算账胶囊。

> 心智模型：**词面增强的核心问题不是「哪个操作最好」，而是「哪些词不能动」。**

## 0 · 复用模块 00 的三重检验

先把模块 00 建立的规则任务与三个检验函数搬过来（本课每个模块都会用）。

In [ ]:
import numpy as np, math, random, collections, itertools
rng = np.random.default_rng(0)

POS_WORDS = {'好吃', '不错', '推荐', '干净', '很好', '满意', '喜欢'}
NEG_WORDS = {'难吃', '差', '脏', '失望', '糟糕'}
NEGATORS  = {'不', '没', '别', '不太', '并不'}
DEGREE    = {'很', '非常', '略', '稍微'}                            # 程度副词
NEUTRAL   = ['这家', '店', '的', '菜', '服务', '环境', '价格', '味道', '朋友', '下次',
             '我们', '昨天', '一起', '去', '吃', '了', '感觉', '整体', '还', '挺']
ENTITIES  = ['海底捞', '西湖', '张三']

PUNCT_SET = {'.', ',', '!', '?', ';', '。', '，', '！', '？', '；'}

def rule_label(tokens, window=3):
    '''规则标签。**算否定词距离时忽略标点** —— 否则插入标点会把否定词推出窗口，
       让 AEDA 这类「不动任何实词」的增强也被误判成破坏了标签。
       这本身是个教训：**保真度检验本身也要设计对。**'''
    tokens = [w for w in tokens if w not in PUNCT_SET]
    score = 0
    for i, t in enumerate(tokens):
        if t in POS_WORDS:
            neg = any(tokens[j] in NEGATORS for j in range(max(0, i-window), i))
            score += -1 if neg else 1
        elif t in NEG_WORDS:
            neg = any(tokens[j] in NEGATORS for j in range(max(0, i-window), i))
            score += 1 if neg else -1
    return 1 if score > 0 else 0

def make_sentence(r, length=10):
    toks = list(r.choice(NEUTRAL, size=length-2, replace=True))
    pos = int(r.integers(1, len(toks)))
    toks.insert(pos, str(r.choice(sorted(POS_WORDS if r.random() < 0.5 else NEG_WORDS))))
    if r.random() < 0.4:
        toks.insert(max(0, pos - int(r.integers(1, 3))), str(r.choice(sorted(NEGATORS))))
    if r.random() < 0.25:
        toks.insert(int(r.integers(0, len(toks))), str(r.choice(ENTITIES)))
    return toks, rule_label(toks)

def fidelity(pairs):
    return 1.0 if not pairs else sum(1 for o, y, a in pairs if rule_label(a) == y) / len(pairs)

def ngrams(t, n):
    return [tuple(t[i:i+n]) for i in range(len(t)-n+1)]

def distinct_n(texts, n=2):
    tot, uniq = 0, set()
    for t in texts:
        g = ngrams(t, n); tot += len(g); uniq.update(g)
    return len(uniq)/tot if tot else 0.0

VOCAB_ALL = sorted(set(NEUTRAL) | POS_WORDS | NEG_WORDS | NEGATORS | set(ENTITIES) | DEGREE
                   | {'本', '餐厅', '菜品', '服务员', '氛围', '收费', '觉得', '总体', '[UNK]',
                      '.', ',', '!', '?', ';'})
V2I = {w: i for i, w in enumerate(VOCAB_ALL)}

def featurize(t):
    x = np.zeros(len(VOCAB_ALL) + 1)
    for w in t:
        if w in V2I: x[V2I[w]] += 1.0
    x[-1] = 1.0
    return x

def train_logreg(X, y, epochs=300, lr=0.3, l2=1e-3, seed=0):
    w = np.random.default_rng(seed).normal(size=X.shape[1]) * 0.01
    for _ in range(epochs):
        p = 1/(1+np.exp(-(X @ w)))
        w -= lr * (X.T @ (p - y)/len(y) + l2*w)
    return w

def build_xy(data):
    return np.stack([featurize(t) for t, _ in data]), np.array([y for _, y in data])

def effectiveness(train_data, aug_data, test_data, seeds=range(8), fixed_steps=True):
    '''固定「总样本数」以避免混淆「增强」与「训练更久」（见讲解）。'''
    Xte, yte = build_xy(test_data)
    diffs, base, aug = [], [], []
    n_total = len(train_data) + len(aug_data)
    ep_base = max(1, round(300 * n_total / max(1, len(train_data)))) if fixed_steps else 300
    for s in seeds:
        Xb, yb = build_xy(train_data)
        wb = train_logreg(Xb, yb, epochs=ep_base, seed=s)
        Xa, ya = build_xy(train_data + aug_data)
        wa = train_logreg(Xa, ya, epochs=300, seed=s)
        b = float(((Xte @ wb > 0).astype(int) == yte).mean())
        a = float(((Xte @ wa > 0).astype(int) == yte).mean())
        base.append(b); aug.append(a); diffs.append(a-b)
    return float(np.mean(base)), float(np.mean(aug)), diffs

TRAIN = [make_sentence(np.random.default_rng(s)) for s in range(200)]
TEST  = [make_sentence(np.random.default_rng(50_000+s)) for s in range(800)]
print(f'训练 {len(TRAIN)} 条, 测试 {len(TEST)} 条')
print('样例:', ' '.join(TRAIN[0][0]), '->', '正面' if TRAIN[0][1] else '负面')
print('✅ 三重检验工具就绪')

## 1 · EDA 四操作：实现与危险性排序

四个操作对「句子结构」的破坏程度不同，所以标签破坏率会差一个量级。
**危险性排序：RD > RS > RI ≈ SR。**

In [ ]:
# 迷你同义词表 —— 刻意埋了陷阱：'好' 的同义词强度不同、'银行' 一词多义
SYNONYMS = {
    '这家': ['本'], '店': ['餐厅'], '菜': ['菜品'], '服务': ['服务员'],
    '环境': ['氛围'], '价格': ['收费'], '感觉': ['觉得'], '整体': ['总体'],
    # 陷阱：情感词的同义替换会改变强度，甚至改变极性判定
    '不错': ['很好', '满意'], '好吃': ['不错'], '差': ['糟糕'],
}
STOPWORDS = {'的', '了', '还', '挺'}

def synonym_replacement(tokens, n, r):
    idx = [i for i, t in enumerate(tokens) if t in SYNONYMS and t not in STOPWORDS]
    if not idx: return list(tokens)
    out = list(tokens)
    for i in r.choice(idx, size=min(n, len(idx)), replace=False):
        out[i] = str(r.choice(SYNONYMS[out[i]]))
    return out

def random_insertion(tokens, n, r):
    out = list(tokens)
    cands = [t for t in tokens if t in SYNONYMS]
    for _ in range(n):
        if not cands: break
        w = str(r.choice(SYNONYMS[str(r.choice(cands))]))
        out.insert(int(r.integers(0, len(out)+1)), w)
    return out

def random_swap(tokens, n, r):
    out = list(tokens)
    for _ in range(n):
        if len(out) < 2: break
        i, j = r.choice(len(out), size=2, replace=False)
        out[i], out[j] = out[j], out[i]
    return out

def random_deletion(tokens, p, r):
    out = [t for t in tokens if r.random() >= p]
    return out if out else list(tokens[:1])

OPS = {
    'SR 同义词替换': lambda t, a, r: synonym_replacement(t, max(1, int(a*len(t))), r),
    'RI 随机插入':   lambda t, a, r: random_insertion(t, max(1, int(a*len(t))), r),
    'RS 随机交换':   lambda t, a, r: random_swap(t, max(1, int(a*len(t))), r),
    'RD 随机删除':   lambda t, a, r: random_deletion(t, a, r),
}

ALPHA = 0.1
print(f"α={ALPHA}")
print(f"{'操作':<16s} {'标签保真度':>11s} {'破坏率':>9s} {'distinct-2':>11s}")
results = {}
for name, fn in OPS.items():
    r = np.random.default_rng(11)
    pairs = [(t, y, fn(t, ALPHA, r)) for t, y in TRAIN]
    f = fidelity(pairs); d = distinct_n([a for _, _, a in pairs], 2)
    results[name] = (f, d)
    print(f'{name:<16s} {f:>11.1%} {1-f:>9.1%} {d:>11.4f}')

break_rates = {k: 1-v[0] for k, v in results.items()}
assert break_rates['RD 随机删除'] > break_rates['SR 同义词替换'], 'RD 应比 SR 危险'
assert break_rates['RS 随机交换'] > 0, 'RS 会破坏语序进而破坏标签'
order = sorted(break_rates, key=break_rates.get, reverse=True)
print(f'\n危险性排序（破坏率从高到低）: {" > ".join(o.split()[0] for o in order)}')
print('✅ 与讲解一致：RD 最危险（直接删信息），SR 最温和（保持槽位）')

### 为什么 RD 特别恶劣：它系统性地污染「关键样本」

RD 破坏标签的概率与「关键词的稀有度」成正比。
**句子里只有一个「不」字时，删掉它的概率就是 p —— 而这个「不」恰恰决定标签。**

In [ ]:
def has_single_negator(tokens):
    return sum(1 for t in tokens if t in NEGATORS) == 1

def break_rate_by_group(op_fn, alpha, data, seed=13):
    r = np.random.default_rng(seed)
    groups = {'含唯一否定词': [], '不含否定词': []}
    for t, y in data:
        aug = op_fn(t, alpha, r)
        key = '含唯一否定词' if has_single_negator(t) else '不含否定词'
        groups[key].append(rule_label(aug) != y)
    return {k: (float(np.mean(v)) if v else 0.0, len(v)) for k, v in groups.items()}

print(f"{'操作':<16s} {'含唯一否定词的破坏率':>22s} {'不含否定词的破坏率':>20s}")
for name, fn in OPS.items():
    g = break_rate_by_group(fn, 0.1, TRAIN)
    print(f'{name:<16s} {g["含唯一否定词"][0]:>21.1%} {g["不含否定词"][0]:>19.1%}')

g_rd = break_rate_by_group(OPS['RD 随机删除'], 0.1, TRAIN)
assert g_rd['含唯一否定词'][0] > g_rd['不含否定词'][0] * 1.5, \
    'RD 对「标签由单个关键词决定」的样本破坏率明显更高'
print(f'\n⚠️  RD 对含唯一否定词的样本破坏率是不含的 '
      f'{g_rd["含唯一否定词"][0]/max(g_rd["不含否定词"][0],1e-9):.1f} 倍。')
print('   **这不是随机噪声，是系统性地污染了最关键的那部分数据。**')
print('   而这类样本恰恰是模型最需要学好的（决策边界就在这里）。')

## 2 · 保护规则：把「不能动的词」标出来

否定词、程度副词、命名实体、数字、以及**用互信息自动挑出的高关键词**。
保护之后，最危险的 RD 也能达到高保真度。

In [ ]:
def mutual_info_keywords(data, top_k=8):
    '''用 |P(y=1|w) - P(y=1)| 自动挑出与标签强相关的词（无需领域知识）。'''
    base = float(np.mean([y for _, y in data]))
    cnt, pos = collections.Counter(), collections.Counter()
    for t, y in data:
        for w in set(t):
            cnt[w] += 1; pos[w] += y
    scores = {w: abs(pos[w]/c - base) for w, c in cnt.items() if c >= 5}
    return set(sorted(scores, key=scores.get, reverse=True)[:top_k])

auto_kw = mutual_info_keywords(TRAIN, top_k=8)
print('互信息自动挑出的关键词:', sorted(auto_kw))
overlap = auto_kw & (POS_WORDS | NEG_WORDS | NEGATORS)
print(f'其中命中真正的情感/否定词: {sorted(overlap)}  ({len(overlap)}/{len(auto_kw)})')
assert len(overlap) >= 3, '自动方法应能挑出多数真正的关键词'
print('✅ 不需要领域知识也能自动找出「不能动的词」')

In [ ]:
def protected_set(extra=frozenset()):
    # 否定词 + 程度副词 + 实体 + **情感词本身** + 互信息自动挑出的关键词
    return NEGATORS | DEGREE | set(ENTITIES) | POS_WORDS | NEG_WORDS | set(extra)

def protected_deletion(tokens, p, r, protect):
    '''受保护的词永不删除。注意保护表要**同时**包含否定词与情感词 ——
       只保护否定词是不够的：删掉唯一的情感词，标签同样会变。'''
    out = [t for t in tokens if (t in protect) or (r.random() >= p)]
    return out if out else list(tokens[:1])

def protected_swap(tokens, n, r, protect):
    '''只交换未受保护的词（保持受保护词的位置）。'''
    out = list(tokens)
    free = [i for i, t in enumerate(out) if t not in protect]
    for _ in range(n):
        if len(free) < 2: break
        i, j = r.choice(free, size=2, replace=False)
        out[i], out[j] = out[j], out[i]
    return out

def protected_synonym(tokens, n, r, protect):
    idx = [i for i, t in enumerate(tokens) if t in SYNONYMS and t not in protect]
    out = list(tokens)
    if not idx: return out
    for i in r.choice(idx, size=min(n, len(idx)), replace=False):
        out[i] = str(r.choice(SYNONYMS[out[i]]))
    return out

PROT = protected_set(auto_kw)
print(f'保护表大小: {len(PROT)} / 词表 {len(VOCAB_ALL)}\n')
print(f"{'操作':<24s} {'保真度':>9s} {'distinct-2':>11s}")
rows = []
for label, fn in [
    ('RD  无保护 p=0.1',      lambda t, r: random_deletion(t, 0.1, r)),
    ('RD  有保护 p=0.1',      lambda t, r: protected_deletion(t, 0.1, r, PROT)),
    ('RS  无保护 n=1',        lambda t, r: random_swap(t, 1, r)),
    ('RS  有保护 n=1',        lambda t, r: protected_swap(t, 1, r, PROT)),
    ('SR  有保护',            lambda t, r: protected_synonym(t, 1, r, PROT)),
]:
    r = np.random.default_rng(17)
    pairs = [(t, y, fn(t, r)) for t, y in TRAIN]
    f, d = fidelity(pairs), distinct_n([a for _, _, a in pairs], 2)
    rows.append((label, f, d))
    print(f'{label:<24s} {f:>9.1%} {d:>11.4f}')

by = {k: (f, d) for k, f, d in rows}
assert by['RD  有保护 p=0.1'][0] > by['RD  无保护 p=0.1'][0], '保护应提高保真度'
assert by['RS  有保护 n=1'][0] > by['RS  无保护 n=1'][0]
assert by['RD  有保护 p=0.1'][0] > 0.97, '保护后 RD 也能达到 97%+ 保真度'
print(f'\n✅ 保护把 RD 的保真度从 {by["RD  无保护 p=0.1"][0]:.1%} 提到 '
      f'{by["RD  有保护 p=0.1"][0]:.1%}，而多样性几乎不损失。')

### 保真-多样权衡前沿：保护得越多，保真越高、多样越低

In [ ]:
def frontier_by_protection(k_values, p=0.15, seed=19):
    out = []
    for k in k_values:
        prot = protected_set(mutual_info_keywords(TRAIN, top_k=k)) if k else set(ENTITIES)
        r = np.random.default_rng(seed)
        pairs = [(t, y, protected_deletion(t, p, r, prot)) for t, y in TRAIN]
        out.append((k, len(prot), fidelity(pairs),
                    distinct_n([a for _, _, a in pairs], 2)))
    return out

print(f"{'top_k':>6s} {'保护表':>7s} {'保真度':>9s} {'distinct-2':>11s}")
front = frontier_by_protection([0, 2, 4, 8, 16, 30])
for k, n, f, d in front:
    print(f'{k:>6d} {n:>7d} {f:>9.1%} {d:>11.4f}')

fids = [f for _, _, f, _ in front]
assert fids[-1] >= fids[0], '保护越多，保真度越高（或至少不降）'
# 「约束下最优」：保真度 >= 0.98 前提下多样性最大的那个点
feasible = [(k, f, d) for k, _, f, d in front if f >= 0.98]
best = max(feasible, key=lambda x: x[2]) if feasible else None
print(f'\n✅ 约束下最优（保真度 ≥ 0.98 且多样性最大）: top_k={best[0]}, '
      f'保真 {best[1]:.1%}, distinct-2 {best[2]:.4f}')
print('   ⚠️ 保护表不能太大：全保护 -> 增强退化成「几乎不变」，多样性趋零、正则化失效。')

## 3 · AEDA：只插标点，保真度接近 100%

Karimi et al. 2021 的洞察：**标点不改变词汇语义**，所以只插标点几乎不可能破坏标签，
却仍能提供表面扰动。想清楚「什么改动不会破坏标签」往往比调参更有价值。

In [ ]:
PUNCT = ['.', ',', '!', '?', ';']

def aeda(tokens, ratio, r):
    '''随机插入 ratio*len 个标点。'''
    out = list(tokens)
    n = max(1, int(ratio * len(tokens)))
    for _ in range(n):
        out.insert(int(r.integers(0, len(out)+1)), str(r.choice(PUNCT)))
    return out

print(f"{'方案':<22s} {'保真度':>9s} {'distinct-2':>11s}")
for label, fn in [('AEDA ratio=0.1', lambda t, r: aeda(t, 0.1, r)),
                  ('AEDA ratio=0.3', lambda t, r: aeda(t, 0.3, r)),
                  ('EDA-RD p=0.1',   lambda t, r: random_deletion(t, 0.1, r))]:
    r = np.random.default_rng(23)
    pairs = [(t, y, fn(t, r)) for t, y in TRAIN]
    print(f'{label:<22s} {fidelity(pairs):>9.1%} '
          f'{distinct_n([a for _, _, a in pairs],2):>11.4f}')

r = np.random.default_rng(23)
pairs_aeda = [(t, y, aeda(t, 0.3, r)) for t, y in TRAIN]
assert fidelity(pairs_aeda) == 1.0, 'AEDA 不改变任何词 -> 规则标签必然不变'
print('\n✅ AEDA 的保真度是**精确的 100%**（不动任何实词，规则标签不可能变）。')
print('   例:', ' '.join(pairs_aeda[0][2]))
print('   在需要「安全的正则化」时，AEDA 常常是比 EDA 更好的默认选择。')

## 4 · α 扫描：先升后急降

$$n_{changed} = \max(1, \lfloor \alpha L \rfloor)$$

原论文实测 α≈0.1 最优、过大时急剧下降。同时看保真度与下游收益。

In [ ]:
def sweep_alpha(alphas, n_aug=4, seed=29):
    out = []
    for a in alphas:
        r = np.random.default_rng(seed)
        aug, pairs = [], []
        for t, y in TRAIN:
            for _ in range(n_aug):
                at = protected_synonym(t, max(1, int(a*len(t))), r, PROT)
                at = protected_swap(at, max(1, int(a*len(at))), r, PROT)
                at = protected_deletion(at, a, r, PROT)
                aug.append((at, y)); pairs.append((t, y, at))
        f = fidelity(pairs); d = distinct_n([x for x, _ in aug], 2)
        b, av, diffs = effectiveness(TRAIN, aug, TEST, seeds=range(6))
        out.append((a, f, d, float(np.mean(diffs)), float(np.std(diffs))))
    return out

print(f"{'α':>6s} {'保真度':>9s} {'distinct-2':>11s} {'Δ准确率':>10s} {'Δ标准差':>9s}")
sw = sweep_alpha([0.02, 0.05, 0.1, 0.2, 0.4])
for a, f, d, dm, ds in sw:
    print(f'{a:>6.2f} {f:>9.1%} {d:>11.4f} {dm:>+10.4f} {ds:>9.4f}')

fids = [x[1] for x in sw]; divs = [x[2] for x in sw]
assert fids == sorted(fids, reverse=True), 'α 越大保真度越低'
assert divs[-1] > divs[0], 'α 越大多样性越高'
print('\n✅ 保真度随 α 单调下降、多样性随 α 单调上升 —— 这就是核心权衡。')
print('   最优 α 在「保真度还能接受」与「多样性足够」之间；原论文实测 0.1 附近。')
print('   ⚠️ 注意 Δ标准差常与 Δ准确率同量级 —— 单点比较不可信（模块 05 给出正确检验）。')

## 5 · 数据量-收益曲线：为什么只有小数据集受益

词面增强提供的是「表面形式不变性」这**一个**归纳偏置。
数据多了，模型本来就学会了这个不变性，增强只剩下噪声。

In [ ]:
def sweep_data_size(sizes, n_aug=4, alpha=0.1, seed=31):
    out = []
    for n in sizes:
        train_n = [make_sentence(np.random.default_rng(s)) for s in range(n)]
        prot = protected_set(mutual_info_keywords(train_n, top_k=8))
        r = np.random.default_rng(seed)
        aug = []
        for t, y in train_n:
            for _ in range(n_aug):
                at = protected_synonym(t, max(1, int(alpha*len(t))), r, prot)
                at = protected_deletion(at, alpha, r, prot)
                aug.append((at, y))
        b, av, diffs = effectiveness(train_n, aug, TEST, seeds=range(6))
        out.append((n, b, av, float(np.mean(diffs)), float(np.std(diffs))))
    return out

print(f"{'训练集大小':>10s} {'baseline':>9s} {'增强后':>8s} {'Δ':>9s} {'Δ标准差':>9s} {'超过噪声?':>10s}")
ds = sweep_data_size([50, 100, 200, 400, 800])
for n, b, av, dm, dsd in ds:
    sig = '✅' if dm > dsd else '❌'
    print(f'{n:>10d} {b:>9.4f} {av:>8.4f} {dm:>+9.4f} {dsd:>9.4f} {sig:>10s}')

deltas = [x[3] for x in ds]
print(f'\n50 条时 Δ={deltas[0]:+.4f} | 800 条时 Δ={deltas[-1]:+.4f}')
assert ds[0][1] < ds[-1][1], 'baseline 准确率应随数据量提高'
print('\n✅ 核心结论：**baseline 随数据量提高，增强的相对收益随之压缩**。')
print('   如果你有十万条标注数据还在纠结要不要做 EDA —— 答案基本是「不要」。')
print('   （本课的合成任务比真实任务简单，收益衰减会更快；但方向与文献一致。）')

## 6 · 在线 vs 离线增强：epoch 混淆与 token dropout

In [ ]:
def token_dropout(tokens, p, r):
    '''在线增强：换成 [UNK] 而非删除 —— 长度与位置不变，比 RD 安全得多。'''
    return [('[UNK]' if r.random() < p else t) for t in tokens]

r = np.random.default_rng(37)
pairs_do = [(t, y, token_dropout(t, 0.1, r)) for t, y in TRAIN]
r = np.random.default_rng(37)
pairs_rd = [(t, y, random_deletion(t, 0.1, r)) for t, y in TRAIN]
print(f'token dropout p=0.1 保真度: {fidelity(pairs_do):.1%}')
print(f'random deletion p=0.1 保真度: {fidelity(pairs_rd):.1%}')
print('（两者破坏标签的机制相同——都可能命中否定词——但 dropout 保持长度与位置）')
assert len(pairs_do[0][2]) == len(pairs_do[0][0]), 'dropout 保持长度'
assert len(pairs_rd[0][2]) <= len(pairs_rd[0][0]), 'deletion 改变长度'

# epoch 混淆：不固定总步数时，「增强」与「训练更久」混在一起
r = np.random.default_rng(41)
aug4 = [(protected_deletion(t, 0.1, r, PROT), y) for t, y in TRAIN for _ in range(4)]
b_fix, a_fix, d_fix = effectiveness(TRAIN, aug4, TEST, seeds=range(6), fixed_steps=True)
b_bad, a_bad, d_bad = effectiveness(TRAIN, aug4, TEST, seeds=range(6), fixed_steps=False)
print(f'\n固定总样本数（公平）  : baseline {b_fix:.4f} -> 增强 {a_fix:.4f}, Δ={np.mean(d_fix):+.4f}')
print(f'不固定（baseline 训得少）: baseline {b_bad:.4f} -> 增强 {a_bad:.4f}, Δ={np.mean(d_bad):+.4f}')
assert b_fix >= b_bad - 1e-9, '公平设置下 baseline 训练量更足、分数不低于不公平设置'
print('\n⚠️  不固定总步数时，baseline 只看了 1/5 的样本 -> Δ 被高估。')
print('✅ 正确做法：**固定总样本数或总步数，而不是 epoch 数**。')
print('   在线增强（dropout / embedding 噪声）天然没有这个问题 —— 数据量不变。')

## ✏️ 练习 1：带保护的 EDA 组合

实现 `eda_protected(tokens, alpha, protect, rng, ops=('SR','RS','RD'))`：
按 `ops` 顺序依次施加受保护版本的操作（SR/RS 用 `max(1,int(alpha*len))` 次，RD 用概率 `alpha`），
返回增强后的 tokens。

In [ ]:
def eda_protected(tokens, alpha, protect, rng, ops=('SR', 'RS', 'RD')):
    # TODO: 依次调用 protected_synonym / protected_swap / protected_deletion
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r = np.random.default_rng(43)
pairs = [(t, y, eda_protected(t, 0.1, PROT, r)) for t, y in TRAIN]
f = fidelity(pairs)
assert f > 0.95, f'带保护的组合应有高保真度，得到 {f:.1%}'
assert distinct_n([a for _, _, a in pairs], 2) > 0, '应产生变化'
# 只做 SR 时保真度应更高（改动更少、更温和）
r = np.random.default_rng(43)
pairs_sr = [(t, y, eda_protected(t, 0.1, PROT, r, ops=('SR',))) for t, y in TRAIN]
assert fidelity(pairs_sr) >= f, '单操作应不低于三操作组合的保真度'
# 受保护的词必须一个都没丢
r = np.random.default_rng(43)
for t, y in TRAIN[:50]:
    aug = eda_protected(t, 0.3, PROT, r, ops=('RD',))
    for w in t:
        if w in PROT:
            assert w in aug, f'受保护词 {w} 不应被删除'
print(f'三操作组合保真度 {f:.1%} | 仅 SR {fidelity(pairs_sr):.1%}')
print('✅ 练习 1 通过：受保护词在最激进的 RD 下也不会丢')

## ✏️ 练习 2：约束下的最优 α

实现 `best_alpha(alphas, min_fidelity, protect, data, seed=0)`：
在「保真度 ≥ min_fidelity」的 α 里返回**多样性最高**的那个；无可行解返回 `None`。

In [ ]:
def best_alpha(alphas, min_fidelity, protect, data, seed=0):
    # TODO: 对每个 α 用 eda_protected 增强一遍，算 (fidelity, distinct_n)；
    #       在可行集里返回 distinct 最大的 α
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
a_strict = best_alpha([0.05, 0.1, 0.2, 0.4], 0.99, PROT, TRAIN)
a_loose  = best_alpha([0.05, 0.1, 0.2, 0.4], 0.90, PROT, TRAIN)
print(f'保真度 ≥ 0.99 -> 最优 α = {a_strict}')
print(f'保真度 ≥ 0.90 -> 最优 α = {a_loose}')
assert a_strict is not None and a_loose is not None
assert a_loose >= a_strict, '更松的保真度约束允许更大的 α（更高多样性）'
assert best_alpha([0.05, 0.1], 1.01, PROT, TRAIN) is None, '不可达时返回 None'
print('✅ 练习 2 通过：这是本课反复出现的「约束下最优」思路 ——')
print('   保真度是硬约束，多样性是目标函数。')

## ✏️ 练习 3：增强预算账

实现 `augmentation_budget(n_train, n_aug, break_rate, n_alpha, n_ops, n_seeds)`：
返回 `{'total_samples':…, 'corrupted':…, 'corrupt_share':…, 'n_runs':…}`。
- `total_samples = n_train * (1 + n_aug)`
- `corrupted = n_train * n_aug * break_rate`（错标样本数）
- `corrupt_share = corrupted / total_samples`
- `n_runs = n_alpha * n_ops * n_seeds`（调参要跑多少次训练）

In [ ]:
def augmentation_budget(n_train, n_aug, break_rate, n_alpha, n_ops, n_seeds):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
b = augmentation_budget(1000, 4, 0.05, n_alpha=3, n_ops=4, n_seeds=5)
assert b['total_samples'] == 5000
assert abs(b['corrupted'] - 200) < 1e-9, '1000×4×5% = 200 条错标数据'
assert abs(b['corrupt_share'] - 0.04) < 1e-9
assert b['n_runs'] == 60
print(f'1000 条 × 4 份 × 破坏率 5%:')
print(f'  总样本 {b["total_samples"]}, 其中错标 {b["corrupted"]:.0f} 条 ({b["corrupt_share"]:.1%})')
print(f'  认真调参需要跑 {b["n_runs"]} 次训练')
# 保护规则把破坏率降到 1% 时
b2 = augmentation_budget(1000, 4, 0.01, 3, 4, 5)
assert b2['corrupted'] < b['corrupted'] / 4
print(f'\n加了保护规则（破坏率 5% -> 1%）: 错标从 {b["corrupted"]:.0f} 降到 {b2["corrupted"]:.0f} 条')
print('✅ 练习 3 通过：「破坏率 5%」= 200 条错标数据被塞进一个 1000 条的训练集 ——')
print('   这就是为什么保真度必须是**硬门槛**而不是软指标。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def eda_protected(tokens, alpha, protect, rng, ops=('SR', 'RS', 'RD')):
    out = list(tokens)
    for op in ops:
        n = max(1, int(alpha * len(out)))
        if op == 'SR':   out = protected_synonym(out, n, rng, protect)
        elif op == 'RS': out = protected_swap(out, n, rng, protect)
        elif op == 'RD': out = protected_deletion(out, alpha, rng, protect)
        elif op == 'RI': out = random_insertion(out, n, rng)
        else: raise ValueError(op)
    return out

In [ ]:
# 练习 2 参考答案
def best_alpha(alphas, min_fidelity, protect, data, seed=0):
    feasible = []
    for a in alphas:
        r = np.random.default_rng(seed)
        pairs = [(t, y, eda_protected(t, a, protect, r)) for t, y in data]
        f = fidelity(pairs)
        d = distinct_n([x for _, _, x in pairs], 2)
        if f >= min_fidelity:
            feasible.append((a, d))
    return max(feasible, key=lambda x: x[1])[0] if feasible else None

In [ ]:
# 练习 3 参考答案
def augmentation_budget(n_train, n_aug, break_rate, n_alpha, n_ops, n_seeds):
    total = n_train * (1 + n_aug)
    corrupted = n_train * n_aug * break_rate
    return {'total_samples': total, 'corrupted': corrupted,
            'corrupt_share': corrupted / total, 'n_runs': n_alpha * n_ops * n_seeds}

---
## 🧪 真实数据胶囊：EDA 原论文的数据量-收益曲线

用 Wei & Zou 2019 报告的量级复现那张关键图，并把「什么时候不该用 EDA」变成一条判据。

In [ ]:
# EDA 论文报告的量级（五个数据集平均，CNN/RNN 模型）
paper = [
    # (训练集比例, baseline 准确率, +EDA 准确率)
    (0.01, 0.700, 0.760),
    (0.05, 0.792, 0.816),
    (0.10, 0.822, 0.840),
    (0.20, 0.845, 0.854),
    (0.50, 0.868, 0.872),
    (1.00, 0.882, 0.885),
]
print(f"{'训练集比例':>10s} {'baseline':>9s} {'+EDA':>7s} {'Δ':>7s} {'相对误差降低':>12s}")
for frac, b, e in paper:
    err_red = (e - b) / (1 - b)
    print(f'{frac:>10.0%} {b:>9.3f} {e:>7.3f} {e-b:>+7.3f} {err_red:>11.1%}')

deltas = [e - b for _, b, e in paper]
assert deltas[0] > deltas[-1] * 5, '1% 数据时的收益应远大于 100% 数据时'
assert deltas == sorted(deltas, reverse=True), '收益应随数据量单调递减'
print(f'\n✅ 1% 数据时 Δ={deltas[0]:+.3f}，100% 数据时 Δ={deltas[-1]:+.3f} —— '
      f'差 {deltas[0]/deltas[-1]:.0f} 倍。')
print('   而 100% 数据时的 +0.003 已经落在种子方差（C49: 2-3 分）之内 —— 不可信。')

SEED_NOISE = 0.02        # 小数据集微调的典型种子标准差（C49 模块 03）
print(f'\n用「Δ 是否超过种子噪声 {SEED_NOISE}」做判据:')
for frac, b, e in paper:
    print(f'  {frac:>5.0%} 数据: Δ={e-b:+.3f} -> {"值得做 ✅" if e-b > SEED_NOISE else "落在噪声内 ❌"}')
worth = [f for f, b, e in paper if e - b > SEED_NOISE]
assert worth and max(worth) <= 0.10, 'EDA 只在很小的数据规模上有可信收益'
print(f'\n✅ 只有训练集 ≤ {max(worth):.0%} 时收益才明显超过噪声。')
print('   **这就是「小数据集才受益」的定量版本。**')

**🧪 胶囊练习**：实现 `should_augment(n_train, expected_delta, seed_noise, labeling_cost_per_sample, eng_hours, hourly_cost)`：
返回 `(是否值得, 增强的工程成本, 同成本能标注的样本数)`。
- 值得的条件：`expected_delta > seed_noise`
- 增强工程成本 = `eng_hours * hourly_cost`
- 同成本能标注 = `int(增强成本 / labeling_cost_per_sample)`

In [ ]:
def should_augment(n_train, expected_delta, seed_noise, labeling_cost_per_sample,
                   eng_hours, hourly_cost):
    # TODO
    raise NotImplementedError

In [ ]:
# 自测
ok, cost, n_label = should_augment(1000, expected_delta=0.03, seed_noise=0.02,
                                  labeling_cost_per_sample=0.5, eng_hours=24, hourly_cost=60)
assert ok is True and abs(cost - 1440) < 1e-9 and n_label == 2880
print(f'预期 Δ=0.03 > 噪声 0.02 -> 值得做')
print(f'但增强的工程成本 ${cost:,.0f} 等价于标注 {n_label:,} 条真实数据')
print(f'（而训练集只有 1000 条 —— 也就是说可以把数据量变成 {1 + n_label/1000:.1f} 倍）')
ok2, _, _ = should_augment(1000, 0.005, 0.02, 0.5, 24, 60)
assert ok2 is False, '效应落在噪声内 -> 不值得'
print('\n✅ 胶囊练习通过：**这就是模块 05 那个必须做的对照** ——')
print('   同样的预算，做增强 vs 直接标注真实数据，哪个划算？')
print('   注意：真实标注成本、工程时间都要按你自己的情况填。')

In [ ]:
# 📖 胶囊参考答案
def should_augment(n_train, expected_delta, seed_noise, labeling_cost_per_sample,
                   eng_hours, hourly_cost):
    cost = eng_hours * hourly_cost
    return (expected_delta > seed_noise, cost, int(cost / labeling_cost_per_sample))

### 小结
- **EDA 四操作的危险性排序 RD > RS > RI ≈ SR**（已量化，破坏率差一个量级）。
- **RD 特别恶劣**：它破坏标签的概率与关键词稀有度成正比 → **系统性污染「标签由单个关键词决定」的关键样本**。
- **保护规则是核心**：否定词 + 程度副词 + 实体 + 数字 + **互信息自动挑出的高关键词**。保护后连 RD 都能到 97%+ 保真度，多样性几乎不损失。
- **保护表不能太大**：全保护 → 增强退化成不变。用「保真度 ≥ 0.98 前提下多样性最大」找最优点。
- **AEDA（只插标点）保真度精确 100%**——想清楚「什么改动不破坏标签」常比调参更有价值。
- **α 的权衡**：保真度单调降、多样性单调升；原论文最优 ≈ 0.1。
- **数据量-收益单调递减**：EDA 只在训练集 ≤ 10% 规模时收益超过种子噪声。
- **固定总样本数而不是 epoch 数**，否则「增强」与「训练更久」混在一起。
- 预算账：破坏率 5% = 200 条错标数据进入 1000 条训练集；认真调参要跑 60+ 次训练，其成本可能能标注几千条真实数据。

下一站：**模块 02 · 回译与释义** —— 上到语义层，保真度更好，但失效方式更隐蔽。